# BERT second opinion: data → fine-tune → calibrate → ONNX (CSC3104 task 7.1–7.3)

Builds the model the `bert` pod serves (`backend/workers/bert`). **Runtime → Change runtime type → T4 GPU**, then
Run all. ~25–40 min on a free T4.

**What it does**
1. Builds windows of up to 5 consecutive **caller** lines from whole conversations, labelled scam / not scam with
   each dataset's own call label (FGRC-SCD + CrossWOZ for Chinese, BothBosu for English). Split by conversation
   group (an FGRC scam and its 无风险 twin always land in the same split).
2. Fine-tunes `distilbert-base-multilingual-cased` (one model for EN + ZH).
3. Fits a temperature on the validation split, so the score is an honest probability (the aggregator averages it
   with the workers' calibrated vote).
4. Exports ONNX, quantises to int8, and checks the int8 file **with the pod's own predict.py** (train = serve).
5. Writes the 3 files the pod needs + `eval_calls.jsonl` for `tools/eval_cascade.py`.

**Before you run**: put these repo files in Drive `MyDrive/CSC3104/bert/repo/` (or let cell 1 clone the repo):
`backend/workers/bert/bert_text.py`, `backend/workers/bert/predict.py`,
`backend/workers/common_zh/gold_test_zh.csv`, `backend/workers/urgency_en/eval/robocall_gold.csv`.
The raw Chinese data is where `zh_data_prep.ipynb` put it (`MyDrive/CSC3104/data/zh/raw/`).

In [ ]:
# 1. Setup
!pip -q install "transformers>=4.44" datasets accelerate onnx onnxruntime==1.18.1 tokenizers opencc-python-reimplemented huggingface_hub
import os, re, sys, json, glob, time, shutil, hashlib, random, subprocess
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/CSC3104'
RAW_ZH = f'{BASE}/data/zh/raw'
WORK = f'{BASE}/bert'                      # everything this notebook writes
OUT = f'{WORK}/model_out'                  # the 3 files for backend/workers/bert/model/
REPO = '/content/repo_files'
os.makedirs(OUT, exist_ok=True); os.makedirs(REPO, exist_ok=True)

GIT_URL = 'https://github.com/notmufliha/CSC3104-CLOUD.git'   # private repo: use https://<token>@github.com/...
GIT_BRANCH = 'bert-cascade'                                   # the branch with backend/workers/bert
NEED = {'bert_text.py': 'backend/workers/bert/bert_text.py', 'predict.py': 'backend/workers/bert/predict.py',
        'gold_test_zh.csv': 'backend/workers/common_zh/gold_test_zh.csv',
        'robocall_gold.csv': 'backend/workers/urgency_en/eval/robocall_gold.csv'}
r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', GIT_BRANCH, GIT_URL, '/content/repo'], capture_output=True)
for name, path in NEED.items():
    src = f'/content/repo/{path}' if r.returncode == 0 else f'{WORK}/repo/{name}'
    assert os.path.exists(src), f'missing {name}: clone failed, so upload it to {WORK}/repo/'
    shutil.copy(src, f'{REPO}/{name}')
print('repo files from', 'git clone' if r.returncode == 0 else 'Drive', '->', sorted(os.listdir(REPO)))
sys.path.insert(0, REPO)
import bert_text                            # the SAME text prep the pod uses
print(bert_text.make_window(['您好，我是新加坡警察部隊的。', '請馬上轉賬']))

SEED = 42
random.seed(SEED); np.random.seed(SEED)
MODEL_NAME = 'distilbert-base-multilingual-cased'
MAX_LEN = 256
FGRC_SCENARIOS_PER_TYPE = 400     # scam + its 无风险 twin per scenario; 9 types -> ~7k dialogues
CROSSWOZ_DIALOGUES = 1200
MAX_WINDOWS_PER_CALL = 6
FIRST_SCAM_LINE = 2               # scam windows must END at caller line >= 3 (index 2): openings are not scams yet
VERSION = 'bert-v1-' + time.strftime('%Y%m%d')

## 2. Conversations → caller lines
One "line" = one caller turn (what the phone's speech recogniser sends per utterance). Very long turns are split
into sentences. Each call keeps a `group`: rows in one group always go to the same split.

In [ ]:
SENT_END = re.compile(r'(?<=[。！？!?；;])\s*')

def turn_lines(turn, max_chars=120):
    t = re.sub(r'\s+', ' ', str(turn).replace('\\n', ' ')).strip()
    if not t:
        return []
    if len(t) <= max_chars:
        return [t]
    out, buf = [], ''
    for s in SENT_END.split(t):
        if buf and len(buf) + len(s) > max_chars:
            out.append(buf); buf = ''
        buf += s
    return out + ([buf] if buf else [])

def fgrc_calls(raw_zh, per_type, seed=SEED):
    files = sorted(glob.glob(f'{raw_zh}/fgrc_scd/FGRC-SCD-dialog/dialog/prompt*_8w.json'))
    assert files, f'no FGRC files under {raw_zh}/fgrc_scd (see zh_data_prep.ipynb)'
    fg = pd.concat([pd.read_json(f).assign(_p=os.path.basename(f)[:7]) for f in files], ignore_index=True)
    fg['scenario'] = 'fgrc_' + fg['_p'].str.replace('prompt', 'p') + '_' + fg['index'].astype(int).astype(str)
    fg['is_scam'] = fg['riskType'] != '无风险'
    pair_type = fg[fg.is_scam].groupby('scenario')['riskType'].first()
    picked = (pair_type.reset_index().groupby('riskType', group_keys=False)
              .apply(lambda d: d.sample(min(len(d), per_type), random_state=seed)))['scenario']
    fg = fg[fg.scenario.isin(set(picked))]
    calls = []
    for _, r in fg.iterrows():
        parts = re.split(r'【(坐席|客户)】', str(r['text']))
        lines = [l for spk, txt in zip(parts[1::2], parts[2::2]) if spk == '坐席' for l in turn_lines(txt)]
        calls.append(dict(call_id=f"{r.scenario}_{'scam' if r.is_scam else 'norisk'}", group=r.scenario, source='fgrc_scd',
                          lang='zh', label=int(r.is_scam), kind=pair_type.get(r.scenario, ''), lines=lines))
    return calls

def crosswoz_calls(raw_zh, n, seed=SEED):
    calls = []
    for f in sorted(glob.glob(f'{raw_zh}/crosswoz/*.json')):
        split = os.path.basename(f).replace('.json', '')
        for did, d in json.load(open(f, encoding='utf-8')).items():
            lines = [l for m in d['messages'] if m['role'] == 'sys' for l in turn_lines(m['content'])]
            calls.append(dict(call_id=f'cw_{split}_{did}', group=f'cw_{split}_{did}', source='crosswoz', lang='zh',
                              label=0, kind='travel_service', lines=lines))
    random.Random(seed).shuffle(calls)
    return calls[:n]

QS_NAME = re.compile(r'(?:^|(?<=\s))([^\s:：，。！？,]{1,12})[:：]')
def quanshiyun_calls(raw_zh, n_per_label=400, seed=SEED):
    """GPL-3.0 data: used ONLY as an out-of-distribution TEST set here, never for training."""
    p = f'{raw_zh}/quanshiyun.parquet'
    if not os.path.exists(p):
        print('no quanshiyun.parquet: skipping the ZH out-of-distribution test'); return []
    q = pd.read_parquet(p)
    q = q[~(q.is_fraud & (q.fraud_speaker.fillna('') == ''))]
    if 'earnings_call' in q:
        q = q[~q.earnings_call.astype(bool)]
    calls = []
    for i, r in q.iterrows():
        text = str(r['input']).replace('\\n', '\n')
        names = [m.group(1) for m in QS_NAME.finditer(text)]
        if not names:
            continue
        first = r.fraud_speaker if r.is_fraud else names[0]          # legit: first speaker = the caller
        rx = re.compile(r'(?:^|(?<=\s))(' + '|'.join(map(re.escape, sorted(set(names), key=len, reverse=True))) + r')[:：]')
        pieces = rx.split(text)
        lines = [l for spk, txt in zip(pieces[1::2], pieces[2::2]) if spk == first for l in turn_lines(txt)]
        if len(lines) >= 2:
            calls.append(dict(call_id=f'qs_{i}', group=f'qs_{i}', source='quanshiyun', lang='zh',
                              label=int(bool(r.is_fraud)), kind='quanshiyun', lines=lines))
    if not calls:
        print('quanshiyun: no usable calls'); return []
    df = pd.DataFrame(calls)
    df = df.groupby('label', group_keys=False).apply(lambda d: d.sample(min(len(d), n_per_label), random_state=seed))
    return df.to_dict('records')

EN_SPK = re.compile(r'(?i)(?:^|\s)(caller|receiver|suspect|innocent|scammer|victim|agent|customer|person ?[ab12])\s*:')
CALLER_TAGS = {'caller', 'suspect', 'scammer', 'agent', 'person a', 'persona', 'person 1', 'person1'}
def en_dialogue_lines(dialogue):
    parts = EN_SPK.split(str(dialogue))
    tags = [t.lower() for t in parts[1::2]]
    if not tags:
        return []
    caller = next((t for t in tags if t in CALLER_TAGS), tags[0])   # who rings first is the caller
    return [l for t, txt in zip(tags, parts[2::2]) if t == caller for l in turn_lines(txt, 300)]

def bothbosu_calls(names=('BothBosu/scam-dialogue', 'BothBosu/single-agent-scam-conversations',
                          'BothBosu/multi-agent-scam-conversation')):
    from datasets import load_dataset
    calls = []
    for name in names:
        try:
            ds = load_dataset(name)
        except Exception as e:
            print(f'skip {name}: {type(e).__name__}'); continue
        for split in ds:
            d = ds[split].to_pandas()
            text_col = next(c for c in ('dialogue', 'conversation', 'text') if c in d)
            label_col = next(c for c in ('labels', 'label', 'is_scam') if c in d)
            ok = 0
            for i, r in d.iterrows():
                lines = en_dialogue_lines(r[text_col])
                if len(lines) >= 2:
                    ok += 1
                    cid = f"{name.split('/')[1]}_{split}_{i}"
                    calls.append(dict(call_id=cid, group=cid, source=name.split('/')[1], lang='en',
                                      label=int(r[label_col]), kind=str(r.get('type', '')), lines=lines))
            print(f'{name} [{split}]: {ok}/{len(d)} dialogues parsed, columns {list(d.columns)}')
            assert ok > 0.8 * len(d), f'{name}: speaker tags not recognised, print one dialogue and fix EN_SPK'
    # the same dialogue can appear in several BothBosu sets: keep one copy
    seen, out = set(), []
    for c in calls:
        key = ' '.join(c['lines'])[:200]
        if key not in seen:
            seen.add(key); out.append(c)
    return out

In [ ]:
calls_zh = fgrc_calls(RAW_ZH, FGRC_SCENARIOS_PER_TYPE) + crosswoz_calls(RAW_ZH, CROSSWOZ_DIALOGUES)
calls_en = bothbosu_calls()
calls_qs = quanshiyun_calls(RAW_ZH)
calls = pd.DataFrame(calls_zh + calls_en)
calls = calls[calls.lines.map(len) >= 2].reset_index(drop=True)
print(calls.groupby(['lang', 'source', 'label']).size().to_string())
print('\nexample caller lines:', calls[calls.label == 1].iloc[0].lines[:4])

## 3. Windows (task 7.1)
A window = caller lines `[i-4 … i]` → `bert_text.make_window` (the pod's function). Scam calls only give windows
that end at caller line 3 or later (openings like "您好，请问是陈先生吗" are not scam evidence). At most 6 windows
per call, evenly spaced, so long calls don't dominate.

In [ ]:
def windows_for_call(c, max_w=MAX_WINDOWS_PER_CALL, first_scam=FIRST_SCAM_LINE):
    lines = c['lines']
    ends = [i for i in range(len(lines)) if c['label'] == 0 or i >= first_scam]
    if len(ends) > max_w:
        ends = sorted({ends[k] for k in np.linspace(0, len(ends) - 1, max_w).round().astype(int)})
    out = []
    for i in ends:
        w = bert_text.make_window(lines[: i + 1])
        short = len(w) < 12 if c['lang'] == 'zh' else len(w.split()) < 5
        if not short:
            out.append(dict(text=w, label=c['label'], lang=c['lang'], source=c['source'], group=c['group'],
                            call_id=c['call_id'], end=i))
    return out

def split_groups(calls, seed=SEED):
    from sklearn.model_selection import GroupShuffleSplit
    g = calls.group.values
    tr, rest = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed).split(calls, groups=g))
    rest_calls = calls.iloc[rest]
    va, te = next(GroupShuffleSplit(1, test_size=0.5, random_state=seed).split(rest_calls, groups=rest_calls.group.values))
    split = pd.Series('train', index=calls.index)
    split.iloc[rest[va]] = 'val'; split.iloc[rest[te]] = 'test'
    return split

calls['split'] = split_groups(calls)
W = pd.DataFrame([w for c in calls.to_dict('records') for w in windows_for_call(c)])
W = W.merge(calls[['call_id', 'split']], on='call_id')
# exact duplicates across splits would leak: keep a text only in its first split (train > val > test)
order = {'train': 0, 'val': 1, 'test': 2}
W = W.sort_values('split', key=lambda s: s.map(order)).drop_duplicates('text').reset_index(drop=True)
assert not set(W[W.split == 'train'].group) & set(W[W.split != 'train'].group), 'group leak'
print(pd.crosstab([W.split, W.lang], W.label, margins=True).to_string())

In [ ]:
# Balance TRAIN only: about as many EN as ZH windows (EN data is smaller), and positives 40-60%.
def balance(train, seed=SEED):
    n_en = (train.lang == 'en').sum()
    zh = train[train.lang == 'zh']
    cap = max(n_en, 8000)                       # ZH may be larger than EN, but not swamp it
    if len(zh) > cap:
        zh = zh.groupby('label', group_keys=False).apply(lambda d: d.sample(int(cap * len(d) / len(zh)), random_state=seed))
    return pd.concat([train[train.lang == 'en'], zh]).sample(frac=1, random_state=seed).reset_index(drop=True)

train = balance(W[W.split == 'train'])
val, test = W[W.split == 'val'].reset_index(drop=True), W[W.split == 'test'].reset_index(drop=True)
print('train', len(train), train.groupby(['lang', 'label']).size().to_dict())
print('val', len(val), '| test', len(test))

In [ ]:
# Label-noise eyeball (for the report): 20 random EARLY positive windows. Count how many show a visible scam cue
# (pressure, money/OTP ask, authority claim, odd request). If fewer than ~75%, raise FIRST_SCAM_LINE to 3 and rerun.
for t in train[(train.label == 1) & (train.end <= 3)].sample(20, random_state=1).text:
    print('-', t[-160:])

## 4. Fine-tune (task 7.2)

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, set_seed)
from sklearn.metrics import roc_auc_score, f1_score, fbeta_score, precision_score, recall_score
set_seed(SEED)
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.truncation_side = 'left'                    # keep the NEWEST lines when a window is too long (pod does the same)
lens = [len(tok(t)['input_ids']) for t in train.text.sample(min(2000, len(train)), random_state=0)]
print('token length p50/p95/p99:', np.percentile(lens, [50, 95, 99]).round())

def to_ds(df):
    return Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True)).map(
        lambda b: tok(b['text'], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=['text'])

def softmax_scam(logits, T=1.0):
    z = np.asarray(logits, dtype=np.float64) / T
    z = np.exp(z - z.max(axis=1, keepdims=True))
    return z[:, 1] / z.sum(axis=1)

def metrics(p):
    prob = softmax_scam(p.predictions); y = p.label_ids
    return {'auc': roc_auc_score(y, prob), 'f1': f1_score(y, prob >= 0.5), 'f2': fbeta_score(y, prob >= 0.5, beta=2)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args = TrainingArguments(output_dir='/content/ckpt', eval_strategy='epoch', save_strategy='epoch',
                         learning_rate=3e-5, per_device_train_batch_size=32, per_device_eval_batch_size=64,
                         num_train_epochs=3, warmup_ratio=0.06, weight_decay=0.01, fp16=True,
                         load_best_model_at_end=True, metric_for_best_model='auc', save_total_limit=1,
                         logging_steps=50, report_to='none', seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=to_ds(train), eval_dataset=to_ds(val),
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
t0 = time.time(); trainer.train(); print(f'trained in {(time.time() - t0) / 60:.1f} min')
trainer.save_model(f'{WORK}/hf'); tok.save_pretrained(f'{WORK}/hf')

## 5. Calibrate (temperature) + evaluate

In [ ]:
from scipy.optimize import minimize_scalar

def fit_temperature(logits, y):
    """T minimising the validation log-loss of softmax(logits / T). T > 1 = the model was over-confident."""
    def nll(logT):
        p = np.clip(softmax_scam(logits, np.exp(logT)), 1e-7, 1 - 1e-7)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method='bounded').x))

def logits_of(df):
    return trainer.predict(to_ds(df)).predictions

L_val = logits_of(val)
T = fit_temperature(L_val, val.label.values)
brier = lambda p, y: float(np.mean((p - y) ** 2))
print(f'temperature T = {T:.3f} | val Brier {brier(softmax_scam(L_val), val.label.values):.4f} -> '
      f'{brier(softmax_scam(L_val, T), val.label.values):.4f}')

def report(name, y, p, thr=0.5):
    yhat = p >= thr
    return dict(set=name, n=len(y), pos=int(y.sum()), auc=round(roc_auc_score(y, p), 3) if 0 < y.sum() < len(y) else None,
                P=round(precision_score(y, yhat, zero_division=0), 3), R=round(recall_score(y, yhat, zero_division=0), 3),
                F1=round(f1_score(y, yhat, zero_division=0), 3), F2=round(fbeta_score(y, yhat, beta=2, zero_division=0), 3),
                brier=round(brier(p, y), 4))

rows = [report('val', val.label.values, softmax_scam(L_val, T))]
p_test = softmax_scam(logits_of(test), T)
rows.append(report('test (all)', test.label.values, p_test))
for lang in ('zh', 'en'):
    m = (test.lang == lang).values
    rows.append(report(f'test {lang}', test.label.values[m], p_test[m]))
# out-of-distribution: quanshiyun calls (real, different style; GPL data, test only)
if calls_qs:
    qs = pd.DataFrame([w for c in calls_qs for w in windows_for_call(c)])
    rows.append(report('OOD zh: quanshiyun windows', qs.label.values, softmax_scam(logits_of(qs), T)))
# single sentences (the system sees 1 line at the start of a call): hand-written SG gold + EN robocall set
gold = pd.read_csv(f'{REPO}/gold_test_zh.csv')
gold_df = pd.DataFrame(dict(text=[bert_text.make_window([t]) for t in gold.text], label=(gold.type == 'scam').astype(int)))
p_gold = softmax_scam(logits_of(gold_df), T)
rows.append(report('ZH gold sentences (scam vs near-miss)', gold_df.label.values, p_gold))
robo = pd.read_csv(f'{REPO}/robocall_gold.csv')
robo_df = pd.DataFrame(dict(text=[bert_text.make_window([t]) for t in robo.text], label=robo.label.astype(int)))
rows.append(report('EN robocall sentences', robo_df.label.values, softmax_scam(logits_of(robo_df), T)))
results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv(f'{WORK}/results_bert.csv', index=False)

In [ ]:
# The cases the LR workers get wrong: does BERT get them right? (slide material: "why a cascade")
probe = ['你好，这里是诊所，打电话确认你星期二的预约。', '今天之内回复我就可以，不急。', '有人冒充星展银行的客服，大家要小心。',
         '我是警察，你必须马上把钱转到安全账户，并告诉我验证码。', 'Hi, this is the clinic calling to confirm your appointment on Tuesday.',
         'This is the police. Transfer the money to the safe account now and read me the OTP.']
pp = softmax_scam(logits_of(pd.DataFrame(dict(text=[bert_text.make_window([t]) for t in probe], label=0))), T)
for t, s in zip(probe, pp):
    print(f'{s:.3f}  {t}')
# reliability plot for the report
import matplotlib.pyplot as plt
pv, yv = softmax_scam(L_val, T), val.label.values
bins = np.linspace(0, 1, 11); idx = np.digitize(pv, bins[1:-1])
xs = [pv[idx == b].mean() for b in range(10) if (idx == b).any()]; ys = [yv[idx == b].mean() for b in range(10) if (idx == b).any()]
plt.figure(figsize=(4, 4)); plt.plot([0, 1], [0, 1], '--', color='grey'); plt.plot(xs, ys, 'o-')
plt.xlabel('predicted P(scam)'); plt.ylabel('actual share of scams'); plt.title(f'BERT reliability (val, T={T:.2f})')
plt.tight_layout(); plt.savefig(f'{WORK}/reliability_bert.png', dpi=150); plt.show()

## 6. ONNX int8 export + check with the pod's own code (task 7.3)

In [ ]:
import inspect
from onnxruntime.quantization import quantize_dynamic, QuantType

class LogitsOnly(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

cpu_model = AutoModelForSequenceClassification.from_pretrained(f'{WORK}/hf').eval().cpu()
dummy = tok(['测试 test | 第二句'], return_tensors='pt')
fp32, int8 = '/content/model_fp32.onnx', f'{OUT}/model.onnx'
kw = {'dynamo': False} if 'dynamo' in inspect.signature(torch.onnx.export).parameters else {}
torch.onnx.export(LogitsOnly(cpu_model), (dummy['input_ids'], dummy['attention_mask']), fp32,
                  input_names=['input_ids', 'attention_mask'], output_names=['logits'],
                  dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'},
                                'logits': {0: 'batch'}}, opset_version=17, **kw)
quantize_dynamic(fp32, int8, weight_type=QuantType.QInt8)
print(f'fp32 {os.path.getsize(fp32) / 1e6:.0f} MB -> int8 {os.path.getsize(int8) / 1e6:.0f} MB')
tok.backend_tokenizer.save(f'{OUT}/tokenizer.json')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

cfg = dict(version=VERSION, model_file='model.onnx', base_model=MODEL_NAME, max_len=MAX_LEN, temperature=round(T, 4),
           scam_index=1, max_turns=bert_text.MAX_TURNS, join=bert_text.JOIN,
           model_sha256=sha256(int8),
           bert_text_sha256=hashlib.sha256(open(f'{REPO}/bert_text.py', 'rb').read().replace(b'\r\n', b'\n')).hexdigest(),
           trained_on=sorted(train.source.unique().tolist()), train_windows=int(len(train)),
           first_scam_line=FIRST_SCAM_LINE + 1, metrics=results.to_dict('records'), date=time.strftime('%Y-%m-%d'))
json.dump(cfg, open(f'{OUT}/bert_config.json', 'w'), ensure_ascii=False, indent=2)

# Parity: score test windows with the POD's predict.py on the int8 file, compare with PyTorch (+T)
os.environ['ORT_THREADS'] = '2'
import importlib, predict; importlib.reload(predict)
predict.load_model(OUT)
sample = test.sample(min(400, len(test)), random_state=0).reset_index(drop=True)
lines_of = lambda text: text.split(bert_text.JOIN)                 # a window's lines (make_window is idempotent)
p_pod = np.array([predict.score(lines_of(t)) for t in sample.text])
p_torch = softmax_scam(logits_of(sample), T)
print(f'pod (int8) vs torch: max |diff| {np.abs(p_pod - p_torch).max():.3f}, mean {np.abs(p_pod - p_torch).mean():.4f}')
print(f'AUC torch {roc_auc_score(sample.label, p_torch):.3f} | pod int8 {roc_auc_score(sample.label, p_pod):.3f}')
ts = []
for t in sample.text[:200]:
    t0 = time.perf_counter(); predict.score(lines_of(t)); ts.append((time.perf_counter() - t0) * 1000)
print(f'CPU latency (2 threads, Colab CPU): p50 {np.median(ts):.0f} ms, p95 {np.percentile(ts, 95):.0f} ms')
assert abs(roc_auc_score(sample.label, p_pod) - roc_auc_score(sample.label, p_torch)) < 0.01, 'int8 lost accuracy'

## 7. Hand-over files
* `MyDrive/CSC3104/bert/model_out/` → copy the 3 files into `backend/workers/bert/model/` (or upload, cell below).
* `eval_calls.jsonl` → `data/eval/eval_calls.jsonl` in the repo, for `tools/eval_cascade.py` (7.5 / 7.12).

In [ ]:
ev = calls[calls.split == 'test'].groupby(['lang', 'label'], group_keys=False).apply(
    lambda d: d.sample(min(len(d), 60), random_state=SEED))
ev_rows = ev[['call_id', 'lang', 'label', 'source', 'lines']].to_dict('records')
ev_rows += [dict(call_id=c['call_id'], lang='zh', label=c['label'], source='quanshiyun_ood', lines=c['lines'])
            for c in pd.DataFrame(calls_qs).sample(min(len(calls_qs), 80), random_state=SEED).to_dict('records')] if calls_qs else []
with open(f'{WORK}/eval_calls.jsonl', 'w', encoding='utf-8') as f:
    for r in ev_rows:
        f.write(json.dumps(dict(r, lines=list(r['lines'])), ensure_ascii=False) + '\n')
print(len(ev_rows), 'held-out calls ->', f'{WORK}/eval_calls.jsonl')
print(os.listdir(OUT), json.dumps({k: cfg[k] for k in ('version', 'temperature', 'model_sha256')}, indent=1))

In [ ]:
# OPTIONAL: upload to a PUBLIC Hugging Face model repo so `docker compose build bert` can download it.
# Needs a free HF account + a WRITE token (huggingface.co/settings/tokens). Prints the MODEL_BASE_URL for .env.
UPLOAD = False
HF_REPO = '<your-hf-username>/csc3104-scam-bert'
if UPLOAD:
    from huggingface_hub import HfApi, login
    login()
    api = HfApi(); api.create_repo(HF_REPO, exist_ok=True, private=False)
    info = api.upload_folder(folder_path=OUT, repo_id=HF_REPO, commit_message=VERSION)
    commit = info.oid if hasattr(info, 'oid') else api.list_repo_commits(HF_REPO)[0].commit_id
    print(f'\nMODEL_BASE_URL=https://huggingface.co/{HF_REPO}/resolve/{commit}')